# Uni-MuMER - Kaggle 2xT4 + DagsHub

Train QLoRA, theo dõi metric trực tiếp bằng MLflow và lưu toàn bộ artifact lên DagsHub.

In [ ]:
# 1. Cấu hình
import json
import os
import sys
import uuid
from pathlib import Path

from kaggle_secrets import UserSecretsClient

PROJECT_DIR = "/kaggle/working/test-unimer"
CONDA_DIR = "/kaggle/working/miniconda"
ENV_DIR = f"{CONDA_DIR}/envs/unimumer"
PYTHON = f"{ENV_DIR}/bin/python"

BASE_YAML_CONFIG = "train/Uni-MuMER-train.yaml"
RUNTIME_YAML_CONFIG = "/kaggle/working/runtime_Uni-MuMER-train.yaml"

YAML_CONFIG = BASE_YAML_CONFIG
NOTEBOOK_PATH = "uni-mumer-kaggle-dagshub v5.ipynb"
OUTPUT_DIR = "saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora"

DAGSHUB_USERNAME = "NhatPot"
DAGSHUB_REPO = "test-unimer"
EXPERIMENT_NAME = "Uni-MuMER-Qwen2.5-VL-3B"
RUN_UUID = uuid.uuid4().hex

# Cho notebook import được module nội bộ trong repo, ví dụ scripts.runtime_yaml
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Cho các lệnh subprocess cũng thấy project
os.environ["PYTHONPATH"] = PROJECT_DIR

# Lấy DagsHub token từ Kaggle Secrets
DAGSHUB_TOKEN = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
if not DAGSHUB_TOKEN:
    raise RuntimeError("Kaggle Secret DAGSHUB_TOKEN is missing")

os.environ.update({
    "PROJECT_DIR": PROJECT_DIR,
    "CONDA_DIR": CONDA_DIR,
    "ENV_DIR": ENV_DIR,
    "PYTHON": PYTHON,
    "BASE_YAML_CONFIG": BASE_YAML_CONFIG,
    "RUNTIME_YAML_CONFIG": RUNTIME_YAML_CONFIG,
    "YAML_CONFIG": YAML_CONFIG,
    "NOTEBOOK_PATH": NOTEBOOK_PATH,
    "OUTPUT_DIR": OUTPUT_DIR,
    "RUN_UUID": RUN_UUID,
    "MLFLOW_TRACKING_URI": f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow",
    "MLFLOW_TRACKING_USERNAME": DAGSHUB_USERNAME,
    "MLFLOW_TRACKING_PASSWORD": DAGSHUB_TOKEN,
    "MLFLOW_EXPERIMENT_NAME": EXPERIMENT_NAME,
    "MLFLOW_FLATTEN_PARAMS": "TRUE",
    "MLFLOW_TAGS": json.dumps({
        "run_uuid": RUN_UUID,
        "source": "kaggle",
        "task": "sft",
        "dataset": "parquet_crohme_train",
        "yaml_config": YAML_CONFIG,
    }),
})

print(f"Run UUID: {RUN_UUID}")
print(f"MLflow: {os.environ['MLFLOW_TRACKING_URI']}")
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"BASE_YAML_CONFIG: {BASE_YAML_CONFIG}")
print(f"NOTEBOOK_PATH: {NOTEBOOK_PATH}")
print(f"PROJECT_DIR in sys.path: {PROJECT_DIR in sys.path}")
print(f"PYTHONPATH: {os.environ.get('PYTHONPATH')}")
print(f"runtime_yaml.py exists: {Path(PROJECT_DIR, 'scripts/runtime_yaml.py').exists()}")

In [ ]:
%%bash
# 2. Tạo môi trường Python 3.10
set -euo pipefail

if [[ ! -x "$PYTHON" ]]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
  bash /tmp/miniconda.sh -b -f -p "$CONDA_DIR"
  rm -f /tmp/miniconda.sh
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
  "$CONDA_DIR/bin/conda" create -n unimumer python=3.10 -y
fi

"$PYTHON" --version

In [ ]:
%%bash
# 3. Lấy source code
set -euo pipefail

if [[ ! -d "$PROJECT_DIR/.git" ]]; then
  git clone https://github.com/NhatPot/test-unimer.git "$PROJECT_DIR"
fi

git -C "$PROJECT_DIR" rev-parse --short HEAD

In [ ]:
# %%bash
# # 3.5 Download test data
# set -e

# cd "$PROJECT_DIR"

# pip install -q gdown

# gdown "1uHT7iOFHGASc9_HB-PhpIuJizlOUOg1L" -O crohme_test_images.zip

# unzip -q crohme_test_images.zip -d "$PROJECT_DIR"

# rm crohme_test_images.zip

In [ ]:
%%bash
# 4. Cài dependency
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python --version
python -m pip install -q -r requirements.txt
python -m pip install -q -e train/LLaMA-Factory
python -c "import torch, mlflow; print('GPU:', torch.cuda.get_device_name(0)); print('MLflow:', mlflow.__version__)"

In [ ]:
# 5. Runtime YAML Override (tùy chọn)
import json
import os

from scripts.runtime_yaml import prepare_runtime_yaml

USE_RUNTIME_YAML_OVERRIDE = True

YAML_OVERRIDES = {
    "dataset": "parquet_crohme_train, parquet_crohme_train_can, parquet_crohme_train_tree, parquet_crohme_train_error_find, parquet_crohme_train_error_fix, parquet_hme100k_train",
    "max_samples": 8000,
    "num_train_epochs": 2,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 64,
    "learning_rate": 1.0e-4,
    "lora_rank": 64,
    "logging_steps": 1,
    "save_steps": 50,
    "eval_steps": 50,
    "save_total_limit": 2,
    "save_only_model": False,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "output_dir": "saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora",

    # Để None nếu không muốn override
    "cutoff_len": None,
    "val_size": None,
    "per_device_eval_batch_size": None,
    "lora_alpha": None,
    "lora_dropout": None,
    "warmup_ratio": None,
    "quantization_bit": None,
    "preprocessing_num_workers": None,
    "dataloader_num_workers": None,
    "bf16": None,
    "fp16": None,
}

YAML_CONFIG, OUTPUT_DIR, YAML_DATA = prepare_runtime_yaml(
    project_dir=PROJECT_DIR,
    base_yaml_config=BASE_YAML_CONFIG,
    runtime_yaml_config=RUNTIME_YAML_CONFIG,
    use_override=USE_RUNTIME_YAML_OVERRIDE,
    overrides=YAML_OVERRIDES,
    strict_keys=True,
)

mlflow_tags = json.loads(os.environ.get("MLFLOW_TAGS", "{}"))
mlflow_tags.update({
    "dataset": str(YAML_DATA.get("dataset", "")),
    "yaml_config": YAML_CONFIG,
    "yaml_override": str(USE_RUNTIME_YAML_OVERRIDE).lower(),
})

os.environ.update({
    "YAML_CONFIG": YAML_CONFIG,
    "OUTPUT_DIR": OUTPUT_DIR,
    "MLFLOW_TAGS": json.dumps(mlflow_tags),
})

print("\nFinal Config:")
print(f"   Dataset: {YAML_DATA.get('dataset', 'N/A')}")
print(f"   Max samples: {YAML_DATA.get('max_samples', 'N/A')}")
print(f"   Epochs: {YAML_DATA.get('num_train_epochs', 'N/A')}")
print(f"   Save/Eval Steps: {YAML_DATA.get('save_steps', 'N/A')}/{YAML_DATA.get('eval_steps', 'N/A')}")
print(f"   Save only model: {YAML_DATA.get('save_only_model', 'N/A')}")
print(f"   Output: {OUTPUT_DIR}")

In [ ]:
%%bash
# 6. Kiểm tra DagsHub trước khi train
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python scripts/dagshub_logger.py check --experiment "$MLFLOW_EXPERIMENT_NAME"

In [ ]:
%%bash
# 7. Training + background checkpoint sync
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

echo "Python: $(command -v python)"
echo "Torchrun: $(command -v torchrun)"
echo "YAML_CONFIG: $YAML_CONFIG"
echo "OUTPUT_DIR: $OUTPUT_DIR"
MPLBACKEND=Agg python scripts/train_with_checkpoint_sync.py \
  --experiment "$MLFLOW_EXPERIMENT_NAME" \
  --run-uuid "$RUN_UUID" \
  --yaml-config "$YAML_CONFIG" \
  --output-dir "$OUTPUT_DIR" \
  --project-dir "$PROJECT_DIR" \
  --run-name "uni-mumer-${RUN_UUID:0:8}" \
  --interval-seconds 180 \
  --stable-seconds 20 \
  --artifact-path checkpoints

In [ ]:
# %%bash
# # 8. Full Benchmark Test (3 CROHME datasets)
# set -euo pipefail
# cd "$PROJECT_DIR"
# source "$CONDA_DIR/bin/activate" unimumer

# # Tìm checkpoint cuối cùng
# LAST_CKPT=$(ls -td "$OUTPUT_DIR"/checkpoint-* 2>/dev/null | head -1)

# if [[ -z "$LAST_CKPT" ]]; then
#   echo "ERROR: No checkpoint found in $OUTPUT_DIR"
#   exit 1
# fi

# echo "Testing with checkpoint: $LAST_CKPT"
# echo "Running full benchmark on 3 CROHME datasets (3,332 samples)..."
# echo ""

# # Chạy full benchmark
# python scripts/kaggle_full_test.py \
#   --base-model Qwen/Qwen2.5-VL-3B-Instruct \
#   --adapter-path "$LAST_CKPT" \
#   --test-datasets crohme_2014 crohme_2016 crohme_2019 \
#   --backup-dir example_data/backup \
#   --base-results-dir example_data/CROHME/results \
#   --output-dir kaggle_test_results \
#   --project-dir "$PROJECT_DIR" \
#   --batch-size 2

# # In summary
# echo ""
# echo "============================================================"
# echo "                    TEST SUMMARY"
# echo "============================================================"
# for dataset in crohme_2014 crohme_2016 crohme_2019; do
#   echo ""
#   echo "=== $dataset ==="
#   if [[ -f "kaggle_test_results/${dataset}_results.txt" ]]; then
#     cat "kaggle_test_results/${dataset}_results.txt" | grep -E "(Mean Edit Score|BLEU-4|Character Error Rate|Exact Match)" | head -4
#   else
#     echo "Results not found"
#   fi
# done

# echo ""
# echo "============================================================"
# echo "Full comparison table:"
# cat kaggle_test_results/comparison_table.txt

In [ ]:
%%bash
# 9. Upload artifacts và test results lên DagsHub
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

# Upload model checkpoint và config
python scripts/dagshub_logger.py upload \
  --experiment "$MLFLOW_EXPERIMENT_NAME" \
  --run-uuid "$RUN_UUID" \
  --config "$YAML_CONFIG" \
  --output-dir "$OUTPUT_DIR" \
  --project-dir "$PROJECT_DIR" \
  --notebook "$NOTEBOOK_PATH"

# Upload test results nếu có
if [[ -d "kaggle_test_results" ]]; then
  echo "Uploading test results to DagsHub..."

  python -c "import os, mlflow; mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI']); mlflow.start_run(run_id=os.environ['RUN_UUID']); mlflow.log_artifacts('kaggle_test_results', artifact_path='test_results'); mlflow.end_run(); print('✓ Test results uploaded')"
fi

In [ ]:
# %%bash
# cd /kaggle/working/test-unimer

# echo "Đang ở:"
# pwd

# echo "Branch hiện tại:"
# git branch --show-current

# echo "Trạng thái trước khi pull:"
# git status --short

# echo "Pull code mới nhất:"
# git pull